In [125]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score

from sklearn.neighbors import NearestNeighbors


In [126]:
# The notebook kernel may start outside the project folder.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [folder / "applied-ml-midterm" for folder in candidates.copy()]
# Local project location, used if the kernel starts in an unrelated folder.
candidates.append(Path("/Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm"))

PROJECT_DIR = next((folder for folder in candidates if (folder / "part1-data").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part1-data. Kernel working directory: {cwd}")

ATTEMPT_DIR = PROJECT_DIR / "prith_attempts" / "part_1"
DATA_DIR = PROJECT_DIR / "part1-data"
print("Loading data from:", DATA_DIR)

part1_train = pd.read_csv(DATA_DIR / "part1_train.csv")
part1_val = pd.read_csv(DATA_DIR / "part1_val.csv")
part1_test = pd.read_csv(DATA_DIR / "part1_test.csv")
pathway_catalog = pd.read_csv(DATA_DIR / "pathway_catalog.csv")

print("Training:", part1_train.shape)
print("Validation:", part1_val.shape)
print("Test:", part1_test.shape)
print("Pathway catalog:", pathway_catalog.shape)

part1_train.head()


Loading data from: /Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm/part1-data
Training: (1171, 166)
Validation: (78, 166)
Test: (312, 161)
Pathway catalog: (2501, 5)


,ID,start_year,criteria,diseases_cleaned,drugs,icdcodes,matched_drugs,drug_targets,drug_actions,drug_pathways,...,emb_145,emb_146,emb_147,emb_148,emb_149,Safety,Funding,Efficacy,Recruitment,Other
0,TR08860,2003,\n Inclusion Criteria:\n\n - ...,"['lung cancer, non-small cell']",['gw572016 (lapatinib)'],"[""['C78.00', 'C78.01', 'C78.02', 'D14.30', 'D1...",NaN,NaN,NaN,NaN,...,0.057633,-0.018126,-0.130373,-0.072231,-0.051629,-100,-100,-100,-100,-100
1,TR03494,2004,\n DISEASE CHARACTERISTICS:\n\n ...,['leukemia'],"['autologous tumor cell vaccine', 'therapeutic...","[""['C95.91', 'C95.92', 'Z80.6', 'Z85.6', 'C90....",NaN,NaN,NaN,NaN,...,-0.018860,-0.077071,-0.059479,0.009044,0.021266,-100,-100,-100,-100,-100
2,TR00772,2006,\n Inclusion Criteria:\n\n - ...,"['clear cell renal cell carcinoma', 'recurrent...",['cediranib maleate'],"[""['C22.0', 'C4A.9', 'C7B.1', 'C4A.0', 'C4A.31...",NaN,NaN,NaN,NaN,...,-0.019414,-0.007497,-0.062808,-0.071619,-0.078172,-100,-100,-100,-100,-100
3,TR08870,2007,\n Inclusion Criteria:\n\n - ...,"['pulmonary disease, chronic obstructive']","['uk-432,097', 'placebo']","[""['J44.9', 'J44.1', 'J44.0']""]",NaN,NaN,NaN,NaN,...,0.014069,-0.006230,-0.081606,-0.031338,-0.063282,-100,-100,-100,-100,-100
4,TR02827,2004,\n Inclusion Criteria:\n\n - ...,"['schizophrenia', 'schizoaffective disorder']",['ziprasidone'],"[""['F20.0', 'F20.1', 'F20.2', 'F20.3', 'F20.5'...",ziprasidone,ADRA1A;ADRA1B;ADRA2A;ADRA2B;ADRA2C;CHRM1;CHRM2...,agonist;antagonist;binder,R-HSA-109582;R-HSA-112314;R-HSA-112315;R-HSA-1...,...,0.093187,-0.043244,-0.077393,-0.110769,0.016161,-100,-100,-100,-100,-100


In [127]:
def create_scaled_embeddings(train_df, val_df, test_df):
    embedding_columns = [f"emb_{i}" for i in range(150)]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[embedding_columns])
    X_val = scaler.transform(val_df[embedding_columns])
    X_test = scaler.transform(test_df[embedding_columns])

    return X_train, X_val, X_test, scaler



In [128]:
def train_kmeans(X_train, n_clusters, n_init=100, random_state=42):
    model = KMeans(
        n_clusters=n_clusters,
        n_init=n_init,
        random_state=random_state,
    )
    model.fit(X_train)
    return model


In [129]:
def inspect_cluster_percentages(model, X_train, train_df, percentage_threshold):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    cluster_ids = model.predict(X_train)
    labeled_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()

    clusters = range(model.n_clusters)
    cluster_reason_mapping = {}
    for cluster in clusters:
        labeled_positions = np.flatnonzero((cluster_ids == cluster) & labeled_mask)
        cluster_reason_mapping[int(cluster)] = []
        if len(labeled_positions) == 0:
            continue

        labeled_rows = train_df.iloc[labeled_positions]
        percentages = labeled_rows[reason_columns].mean() * 100
        cluster_reason_mapping[int(cluster)] = [
            reason for reason in reason_columns
            if percentages[reason] >= percentage_threshold
        ]

    return cluster_reason_mapping


In [130]:
def label_unlabeled_points(model, X_train, train_df, cluster_reason_mapping):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    known_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    unlabeled_mask = train_df[reason_columns].eq(-100).all(axis=1).to_numpy()

    result = train_df.copy()
    result["label_source"] = np.where(known_mask, "observed", "unresolved").astype(object)
    cluster_ids = model.predict(X_train)
    reason_positions = result.columns.get_indexer(reason_columns)

    for position in np.flatnonzero(unlabeled_mask):
        reasons = cluster_reason_mapping.get(int(cluster_ids[position]), [])
        if not reasons:
            continue
        result.iloc[position, reason_positions] = [
            int(reason in reasons) for reason in reason_columns
        ]
        result.iloc[position, result.columns.get_loc("label_source")] = "cluster_prediction"

    return result


In [131]:
def train_logistic_regression(X_train, train_df, C=1.0, max_iter=500,
                              random_state=42):
    """Train five classifiers using observed and recovered labels equally."""
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    labeled_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    if not labeled_mask.any():
        raise ValueError("No rows have five known binary reason labels for training.")

    y_train = train_df.loc[labeled_mask, reason_columns].to_numpy(dtype=int)
    model = OneVsRestClassifier(
        LogisticRegression(
            C=C,
            max_iter=max_iter,
            solver="liblinear",
            random_state=random_state,
        )
    )
    model.fit(X_train[labeled_mask], y_train)
    return model


In [132]:
def predict_reason_labels(model, X):
    """Predict binary reasons, choosing the highest-probability reason if all are zero."""
    predictions = np.asarray(model.predict(X)).copy()
    empty_rows = ~predictions.any(axis=1)
    if empty_rows.any():
        probabilities = model.predict_proba(X[empty_rows])
        best_reasons = np.asarray(probabilities).argmax(axis=1)
        predictions[np.flatnonzero(empty_rows), best_reasons] = 1
    return predictions


In [133]:
def evaluate_reason_model(model, X_val, val_df):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    y_true = val_df[reason_columns].to_numpy(dtype=int)
    y_pred = predict_reason_labels(model, X_val)
    per_reason_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

    return pd.Series(
        [
            f1_score(y_true, y_pred, average="samples", zero_division=0),
            f1_score(y_true, y_pred, average="macro", zero_division=0),
            *per_reason_f1,
        ],
        index=["Sample-wise F1", "Macro F1", *reason_columns],
        name="Validation F1",
    )


In [134]:
def create_pca_embeddings(X_train, n_components=50):
    """Fit PCA on scaled training embeddings and return embeddings plus fitted PCA."""
    pca = PCA(n_components=n_components, svd_solver="full")
    X_train_pca = pca.fit_transform(X_train)
    return X_train_pca, pca


In [135]:
def create_kernel_pca_embeddings(X_train, n_components=50, kernel="rbf",
                                 gamma=None, random_state=42):
    """Fit kernel PCA on scaled training embeddings; return embeddings and transformer."""
    effective_gamma = 1 / X_train.shape[1] if gamma is None else gamma
    kernel_pca = KernelPCA(
        n_components=n_components,
        kernel=kernel,
        gamma=effective_gamma,
        eigen_solver="arpack",
        random_state=random_state,
    )
    X_train_kernel_pca = kernel_pca.fit_transform(X_train)
    return X_train_kernel_pca, kernel_pca


In [136]:
# Run the imports, data-loading, and function-definition cells first.
PERCENTAGE_THRESHOLD = 50
RANDOM_STATE = 42  # Kernel PCA and logistic regression stay fixed.
KMEANS_RANDOM_STATE = 42
N_INIT = 100
KERNEL_PCA_COMPONENTS = 50
KERNEL = "rbf"
KERNEL_GAMMA = 1 / 75

# Scaling is shared across runs and fitted on training embeddings only.
X_train, X_val, X_test, scaler = create_scaled_embeddings(
    part1_train, part1_val, part1_test
)

# Kernel PCA is used only for clustering and label recovery.
X_train_cluster, kernel_pca_model = create_kernel_pca_embeddings(
    X_train, n_components=KERNEL_PCA_COMPONENTS, kernel=KERNEL,
    gamma=KERNEL_GAMMA, random_state=RANDOM_STATE,
)
print(f"Clustering uses {X_train_cluster.shape[1]} kernel PCA components.")
print(f"Kernel: {kernel_pca_model.kernel}; gamma: {kernel_pca_model.gamma:.6g}")

# The baseline and augmented classifiers both use the original scaled embeddings.
baseline_model = train_logistic_regression(
    X_train, part1_train, C=1.0, max_iter=500, random_state=RANDOM_STATE,
)
baseline_validation_metrics = evaluate_reason_model(baseline_model, X_val, part1_val)

pipeline_results = {}
# Compare every cluster count from 2 through 30.
for n_clusters in range(2, 31):
    kmeans_model = train_kmeans(
        X_train_cluster, n_clusters=n_clusters,
        n_init=N_INIT, random_state=KMEANS_RANDOM_STATE,
    )
    cluster_reason_mapping = inspect_cluster_percentages(
        kmeans_model, X_train_cluster, part1_train,
        percentage_threshold=PERCENTAGE_THRESHOLD,
    )
    part1_train_augmented = label_unlabeled_points(
        kmeans_model, X_train_cluster, part1_train, cluster_reason_mapping
    )
    reason_model = train_logistic_regression(
        X_train, part1_train_augmented,
        C=1.0, max_iter=500, random_state=RANDOM_STATE,
    )
    validation_metrics = evaluate_reason_model(reason_model, X_val, part1_val)

    pipeline_results[n_clusters] = {
        "kmeans_random_state": KMEANS_RANDOM_STATE,
        "n_init": N_INIT,
        "kernel_pca_model": kernel_pca_model,
        "kmeans_model": kmeans_model,
        "cluster_reason_mapping": cluster_reason_mapping,
        "part1_train_augmented": part1_train_augmented,
        "reason_model": reason_model,
        "validation_metrics": validation_metrics,
        "label_source_counts": part1_train_augmented["label_source"].value_counts(),
    }


Clustering uses 50 kernel PCA components.
Kernel: rbf; gamma: 0.0133333


In [137]:
validation_scores = pd.DataFrame.from_dict(
    {
        n_clusters: run["validation_metrics"]
        for n_clusters, run in pipeline_results.items()
    },
    orient="index",
).sort_index()
validation_scores.index.name = "Number of clusters"
validation_scores = validation_scores[
    ["Sample-wise F1", "Macro F1", "Safety", "Funding", "Efficacy", "Recruitment", "Other"]
]
# Percentage of all training trials still unlabeled after recovery.
reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
validation_scores["Percent unlabeled"] = pd.Series({
    n_clusters: (
        run["part1_train_augmented"][reason_columns].eq(-100).all(axis=1).mean() * 100
    )
    for n_clusters, run in pipeline_results.items()
})
validation_scores["Delta vs baseline"] = (
    validation_scores["Sample-wise F1"] - baseline_validation_metrics["Sample-wise F1"]
)
print(f"Observed-label-only baseline sample-wise F1: {baseline_validation_metrics['Sample-wise F1']:.4f}")
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(validation_scores.round(4))

# Select using the original scores, not their rounded display values.
samplewise_f1_scores = validation_scores["Sample-wise F1"]
best_samplewise_f1 = samplewise_f1_scores.max()
best_cluster_counts = samplewise_f1_scores.index[
    samplewise_f1_scores == best_samplewise_f1
].tolist()
print("Best cluster count(s):", best_cluster_counts)
print(f"Highest validation sample-wise F1: {best_samplewise_f1:.4f}")
# Selecting among 29 settings on 78 validation trials can overfit validation.
# Compare the selected result with the supervised baseline in the report.


Observed-label-only baseline sample-wise F1: 0.8782


,Sample-wise F1,Macro F1,Safety,Funding,Efficacy,Recruitment,Other,Percent unlabeled,Delta vs baseline
Number of clusters,,,,,,,,,
2,0.6496,0.2351,0.2000,0.0000,0.0000,0.7937,0.1818,0.0000,-0.2286
3,0.6496,0.2351,0.2000,0.0000,0.0000,0.7937,0.1818,0.0000,-0.2286
4,0.8594,0.6392,0.6957,0.2222,0.7500,0.9899,0.5385,66.5243,-0.0188
5,0.8594,0.6392,0.6957,0.2222,0.7500,0.9899,0.5385,66.5243,-0.0188
6,0.8628,0.6683,0.7500,0.2500,0.8235,0.9796,0.5385,59.0948,-0.0154
7,0.6581,0.2654,0.2000,0.0000,0.0000,0.7937,0.3333,0.0000,-0.2201
8,0.8581,0.6741,0.8000,0.2857,0.7857,0.9608,0.5385,50.8967,-0.0201
9,0.8581,0.6753,0.8000,0.2857,0.7586,0.9608,0.5714,51.0675,-0.0201
10,0.8769,0.6867,0.8182,0.2222,0.9032,0.9899,0.5000,49.6157,-0.0013


Best cluster count(s): [28]
Highest validation sample-wise F1: 0.8979


In [138]:
# Reuse the labels already recovered for K=14; do not rerun clustering.
# Smaller C means stronger L2 regularization. C=1 is our current control.
REGULARIZATION_CLUSTER_COUNT = 14
C_VALUES = [0.01, 0.1, 1.0, 10.0]

if "pipeline_results" not in globals() or REGULARIZATION_CLUSTER_COUNT not in pipeline_results:
    raise ValueError("Run the cluster-count sweep first so pipeline_results contains K=14.")

regularization_training_data = pipeline_results[REGULARIZATION_CLUSTER_COUNT]["part1_train_augmented"]
regularization_results = {}

for c_value in C_VALUES:
    augmented_regularization_model = train_logistic_regression(
        X_train, regularization_training_data,
        C=c_value, max_iter=500, random_state=RANDOM_STATE,
    )
    augmented_regularization_metrics = evaluate_reason_model(
        augmented_regularization_model, X_val, part1_val
    )
    baseline_regularization_model = train_logistic_regression(
        X_train, part1_train,
        C=c_value, max_iter=500, random_state=RANDOM_STATE,
    )
    baseline_regularization_metrics = evaluate_reason_model(
        baseline_regularization_model, X_val, part1_val
    )

    regularization_results[float(c_value)] = {
        "augmented_model": augmented_regularization_model,
        "validation_metrics": augmented_regularization_metrics,
        "baseline_model": baseline_regularization_model,
        "baseline_validation_metrics": baseline_regularization_metrics,
        "cluster_count": REGULARIZATION_CLUSTER_COUNT,
        "C": float(c_value),
        "penalty": "l2",
        "max_iter": 500,
        "random_state": RANDOM_STATE,
    }


In [139]:
regularization_scores = pd.DataFrame.from_dict(
    {
        c_value: run["validation_metrics"]
        for c_value, run in regularization_results.items()
    },
    orient="index",
).sort_index()
regularization_scores.index.name = "C"
regularization_reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
regularization_scores = regularization_scores[
    ["Sample-wise F1", "Macro F1", *regularization_reason_columns]
]
# The recovered labels are fixed, so this percentage is the same for every C.
regularization_scores["Percent unlabeled"] = (
    regularization_training_data[regularization_reason_columns].eq(-100).all(axis=1).mean() * 100
)
regularization_scores["Baseline sample-wise F1"] = pd.Series({
    c_value: run["baseline_validation_metrics"]["Sample-wise F1"]
    for c_value, run in regularization_results.items()
})
regularization_scores["Delta vs baseline"] = (
    regularization_scores["Sample-wise F1"] - regularization_scores["Baseline sample-wise F1"]
)
print("Fixed recovered labels from cluster count:", REGULARIZATION_CLUSTER_COUNT)
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(regularization_scores.round(4))

best_regularization_sample_score = regularization_scores["Sample-wise F1"].max()
best_regularization_macro_score = regularization_scores["Macro F1"].max()
print("Best sample-wise F1 C value(s):", regularization_scores.index[
    regularization_scores["Sample-wise F1"] == best_regularization_sample_score
].tolist())
print("Best macro F1 C value(s):", regularization_scores.index[
    regularization_scores["Macro F1"] == best_regularization_macro_score
].tolist())
# Review all class scores before selecting a regularization model for export.


Fixed recovered labels from cluster count: 14


,Sample-wise F1,Macro F1,Safety,Funding,Efficacy,Recruitment,Other,Percent unlabeled,Baseline sample-wise F1,Delta vs baseline
C,,,,,,,,,,
0.01,0.8175,0.5887,0.5143,0.2308,0.7000,0.9691,0.5294,44.4065,0.7201,0.0974
0.10,0.8739,0.6477,0.6667,0.1818,0.8485,0.9899,0.5517,44.4065,0.8534,0.0205
1.00,0.8949,0.7221,0.7826,0.3333,0.9333,0.9899,0.5714,44.4065,0.8782,0.0167
10.00,0.8782,0.6867,0.8000,0.2857,0.8966,0.9899,0.4615,44.4065,0.8782,0.0000


Best sample-wise F1 C value(s): [1.0]
Best macro F1 C value(s): [1.0]


In [140]:
def recover_labels_with_neighbors(model, X_cluster, train_df, n_neighbors=3,
                                  cluster_score_weight=0.5, percentage_threshold=50,
                                  X_neighbors=None):
    """Blend raw cluster rates with votes from originally labeled neighbors.

    Cluster assignments use X_cluster; neighbor distances use X_neighbors.
    If omitted, X_neighbors defaults to X_cluster. Both arrays must contain
    the same training trials in the same row order.
    A cluster without known labels uses neighbor votes alone. Empty predicted
    reason sets stay unresolved; originally observed labels are never changed.
    """
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    X_cluster = np.asarray(X_cluster)
    if X_cluster.ndim != 2 or len(X_cluster) != len(train_df):
        raise ValueError("Embeddings and training rows must have matching order and length.")
    X_neighbors = X_cluster if X_neighbors is None else np.asarray(X_neighbors)
    if X_neighbors.ndim != 2 or len(X_neighbors) != len(train_df):
        raise ValueError("Neighbor embeddings must have the same training row order and length.")
    if not 0 <= cluster_score_weight <= 1 or not 0 <= percentage_threshold <= 100:
        raise ValueError("Cluster weight must be in [0, 1] and threshold in [0, 100].")

    known_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    unknown_mask = train_df[reason_columns].eq(-100).all(axis=1).to_numpy()
    if not (known_mask | unknown_mask).all():
        raise ValueError("Expected fully observed or fully masked reason-label rows.")
    if not isinstance(n_neighbors, (int, np.integer)) or not 1 <= n_neighbors <= known_mask.sum():
        raise ValueError("n_neighbors must be a positive integer no larger than the labeled set.")

    result = train_df.copy()
    result["label_source"] = np.where(known_mask, "observed", "unresolved").astype(object)
    known_positions = np.flatnonzero(known_mask)
    unknown_positions = np.flatnonzero(unknown_mask)
    if len(unknown_positions) == 0:
        return result, pd.DataFrame()

    cluster_ids = model.predict(X_cluster)
    y_known = train_df.iloc[known_positions][reason_columns].to_numpy(dtype=int)
    cluster_rates = {}
    cluster_support = {}
    for cluster in np.unique(cluster_ids):
        labeled_positions = np.flatnonzero((cluster_ids == cluster) & known_mask)
        cluster_support[int(cluster)] = len(labeled_positions)
        if len(labeled_positions):
            cluster_rates[int(cluster)] = train_df.iloc[labeled_positions][reason_columns].mean().to_numpy()

    # Fit the neighbor search only on the original known-label trials.
    neighbors = NearestNeighbors(n_neighbors=n_neighbors)
    neighbors.fit(X_neighbors[known_positions])
    distances, neighbor_indices = neighbors.kneighbors(X_neighbors[unknown_positions])
    neighbor_scores = y_known[neighbor_indices].mean(axis=1)
    reason_positions = result.columns.get_indexer(reason_columns)
    source_position = result.columns.get_loc("label_source")
    audit_rows = []

    for row_number, position in enumerate(unknown_positions):
        cluster = int(cluster_ids[position])
        rates = cluster_rates.get(cluster)
        if rates is None:
            scores = neighbor_scores[row_number]
        else:
            scores = (cluster_score_weight * rates
                      + (1 - cluster_score_weight) * neighbor_scores[row_number])
        recovered = (scores >= percentage_threshold / 100).astype(int)
        if recovered.any():
            result.iloc[position, reason_positions] = recovered
            result.iloc[position, source_position] = "cluster_knn_prediction"

        neighbor_positions = known_positions[neighbor_indices[row_number]]
        audit_rows.append({
            "row_position": int(position),
            "ID": train_df.iloc[position]["ID"],
            "cluster": cluster,
            "known_cluster_members": cluster_support[cluster],
            "used_neighbor_only": rates is None,
            "neighbor_IDs": train_df.iloc[neighbor_positions]["ID"].tolist(),
            "neighbor_distances": distances[row_number].tolist(),
            **{f"cluster_rate_{reason}": (np.nan if rates is None else rates[j])
               for j, reason in enumerate(reason_columns)},
            **{f"neighbor_rate_{reason}": neighbor_scores[row_number, j]
               for j, reason in enumerate(reason_columns)},
            **{f"score_{reason}": scores[j] for j, reason in enumerate(reason_columns)},
            "recovered_reasons": [reason for reason, value in zip(reason_columns, recovered) if value],
            "label_source": result.iloc[position, source_position],
        })

    return result, pd.DataFrame(audit_rows)


In [141]:
# Controlled experiment: neighbors in original scaled embeddings; clusters stay in kernel PCA.
NEIGHBOR_CLUSTER_COUNT = 14
NEIGHBOR_COUNT = 3
CLUSTER_SCORE_WEIGHT = 0.5
NEIGHBOR_PERCENTAGE_THRESHOLD = 50

if "pipeline_results" not in globals() or NEIGHBOR_CLUSTER_COUNT not in pipeline_results:
    raise ValueError("Run the cluster-count sweep first so pipeline_results contains K=14.")

neighbor_reference_run = pipeline_results[NEIGHBOR_CLUSTER_COUNT]
neighbor_augmented_train, neighbor_audit = recover_labels_with_neighbors(
    neighbor_reference_run["kmeans_model"], X_train_cluster, part1_train,
    n_neighbors=NEIGHBOR_COUNT,
    cluster_score_weight=CLUSTER_SCORE_WEIGHT,
    percentage_threshold=NEIGHBOR_PERCENTAGE_THRESHOLD,
    X_neighbors=X_train,
)
# The final classifier still uses the original 150 scaled embedding dimensions.
neighbor_reason_model = train_logistic_regression(
    X_train, neighbor_augmented_train, C=1.0, max_iter=500, random_state=RANDOM_STATE
)
neighbor_validation_metrics = evaluate_reason_model(neighbor_reason_model, X_val, part1_val)
neighbor_experiment = {
    "augmented_training": neighbor_augmented_train,
    "audit": neighbor_audit,
    "reason_model": neighbor_reason_model,
    "validation_metrics": neighbor_validation_metrics,
    "settings": {
        "cluster_count": NEIGHBOR_CLUSTER_COUNT,
        "n_neighbors": NEIGHBOR_COUNT,
        "cluster_score_weight": CLUSTER_SCORE_WEIGHT,
        "percentage_threshold": NEIGHBOR_PERCENTAGE_THRESHOLD,
        "neighbor_space": "original 150-dimensional scaled embeddings",
        "cluster_space": "existing kernel PCA embeddings",
        "C": 1.0,
        "random_state": RANDOM_STATE,
    },
}

print(f"Clustering dimensions: {X_train_cluster.shape[1]}; neighbor dimensions: {X_train.shape[1]}")


Clustering dimensions: 50; neighbor dimensions: 150


In [142]:
neighbor_reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
neighbor_comparison = pd.DataFrame({
    "Cluster only": neighbor_reference_run["validation_metrics"],
    "Cluster + neighbors": neighbor_experiment["validation_metrics"],
}).T
neighbor_comparison = neighbor_comparison[
    ["Sample-wise F1", "Macro F1", *neighbor_reason_columns]
]
neighbor_training_sets = {
    "Cluster only": neighbor_reference_run["part1_train_augmented"],
    "Cluster + neighbors": neighbor_augmented_train,
}
for method, augmented_data in neighbor_training_sets.items():
    neighbor_comparison.loc[method, "Percent unlabeled"] = (
        augmented_data[neighbor_reason_columns].eq(-100).all(axis=1).mean() * 100
    )
    originally_masked = part1_train[neighbor_reason_columns].eq(-100).all(axis=1).to_numpy()
    neighbor_comparison.loc[method, "Recovered Funding positives"] = int(
        augmented_data.iloc[np.flatnonzero(originally_masked)]["Funding"].eq(1).sum()
    )
neighbor_comparison["Baseline sample-wise F1"] = baseline_validation_metrics["Sample-wise F1"]
neighbor_comparison["Delta vs baseline"] = (
    neighbor_comparison["Sample-wise F1"] - neighbor_comparison["Baseline sample-wise F1"]
)
with pd.option_context("display.max_columns", None):
    display(neighbor_comparison.round(4))

# Inspect the exact originally labeled neighbors and all five scores per trial.
display(neighbor_audit.head())
# Export remains on the explicitly selected cluster-only reference until a model is chosen.


,Sample-wise F1,Macro F1,Safety,Funding,Efficacy,Recruitment,Other,Percent unlabeled,Recovered Funding positives,Baseline sample-wise F1,Delta vs baseline
Cluster only,0.8949,0.7221,0.7826,0.3333,0.9333,0.9899,0.5714,44.4065,0.0,0.8782,0.0167
Cluster + neighbors,0.8162,0.6395,0.8235,0.4000,0.6364,0.9091,0.4286,12.1264,5.0,0.8782,-0.0620


,row_position,ID,cluster,known_cluster_members,used_neighbor_only,neighbor_IDs,neighbor_distances,cluster_rate_Safety,cluster_rate_Funding,cluster_rate_Efficacy,...,neighbor_rate_Efficacy,neighbor_rate_Recruitment,neighbor_rate_Other,score_Safety,score_Funding,score_Efficacy,score_Recruitment,score_Other,recovered_reasons,label_source
0,0,TR08860,10,74,False,"[TR00079, TR01103, TR04537]","[14.938558056625524, 15.100162172254118, 15.14...",0.216216,0.094595,0.094595,...,0.333333,0.333333,0.333333,0.108108,0.047297,0.213964,0.369369,0.322072,[],unresolved
1,1,TR03494,10,74,False,"[TR04020, TR02607, TR10318]","[11.387095677467794, 12.07313577415366, 12.073...",0.216216,0.094595,0.094595,...,0.000000,1.000000,0.000000,0.108108,0.047297,0.047297,0.702703,0.155405,[Recruitment],cluster_knn_prediction
2,2,TR00772,2,11,False,"[TR03598, TR05495, TR10054]","[9.062072817663953, 9.062074362447019, 9.06207...",0.000000,0.090909,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.045455,0.000000,1.000000,0.000000,[Recruitment],cluster_knn_prediction
3,3,TR08870,10,74,False,"[TR10054, TR05495, TR03598]","[13.98623822659524, 13.986251942235791, 13.986...",0.216216,0.094595,0.094595,...,0.000000,1.000000,0.000000,0.108108,0.047297,0.047297,0.702703,0.155405,[Recruitment],cluster_knn_prediction
4,4,TR02827,3,2,False,"[TR03849, TR08043, TR10054]","[2.954299978153866, 4.964369776291255, 14.4464...",0.000000,0.000000,0.000000,...,0.000000,0.333333,0.666667,0.000000,0.000000,0.000000,0.166667,0.833333,[Other],cluster_knn_prediction


In [143]:
class SelectiveReasonClassifier:
    """Keep five ordered binary estimators compatible with existing evaluation."""
    def __init__(self, estimators):
        self.estimators_ = list(estimators)

    def predict(self, X):
        return np.column_stack([model.predict(X) for model in self.estimators_])

    def predict_proba(self, X):
        return np.column_stack([model.predict_proba(X)[:, 1] for model in self.estimators_])


def fit_selective_reason_model(reference_model, X_train, train_df, reference_augmented_df,
                               knn_recovery_reasons=("Funding",), n_neighbors=3,
                               percentage_threshold=50, C=1.0, random_state=42):
    """Retrain only selected reasons on KNN-recovered binary labels.

    Neighbors are originally labeled trials in X_train (scaled150 embeddings).
    Both positive and negative labels are recovered for every masked trial.
    Other reasons reuse the exact fitted reference estimators, without refitting.
    """
    reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    X_train = np.asarray(X_train)
    if X_train.ndim != 2 or len(X_train) != len(train_df):
        raise ValueError("Training embeddings and trial rows must match in order and length.")
    if not reference_augmented_df["ID"].reset_index(drop=True).equals(train_df["ID"].reset_index(drop=True)):
        raise ValueError("Reference augmented data must contain the same trials in the same order.")
    if not set(knn_recovery_reasons).issubset(reasons) or not knn_recovery_reasons:
        raise ValueError("Choose at least one of the five known reasons for KNN recovery.")
    if not 0 <= percentage_threshold <= 100:
        raise ValueError("percentage_threshold must be in [0, 100].")
    if len(reference_model.estimators_) != len(reasons):
        raise ValueError("Reference estimators must follow the five-reason column order.")

    known = train_df[reasons].isin([0, 1]).all(axis=1).to_numpy()
    masked = train_df[reasons].eq(-100).all(axis=1).to_numpy()
    if not (known | masked).all():
        raise ValueError("Expected fully observed or fully masked label rows.")
    if not isinstance(n_neighbors, (int, np.integer)) or not 1 <= n_neighbors <= known.sum():
        raise ValueError("n_neighbors must not exceed the original labeled set size.")
    if not np.array_equal(reference_augmented_df.loc[known, reasons].to_numpy(),
                          train_df.loc[known, reasons].to_numpy()):
        raise ValueError("Reference data changed originally observed labels.")

    known_positions = np.flatnonzero(known)
    masked_positions = np.flatnonzero(masked)
    y_known = train_df.loc[known, reasons].to_numpy(dtype=int)
    neighbors = NearestNeighbors(n_neighbors=n_neighbors).fit(X_train[known])
    if len(masked_positions):
        neighbor_indices = neighbors.kneighbors(X_train[masked], return_distance=False)
        vote_rates = y_known[neighbor_indices].mean(axis=1)
    else:
        vote_rates = np.empty((0, len(reasons)))

    estimators = list(reference_model.estimators_)
    lineage = train_df[["ID"]].copy()
    coverage_rows = []
    for j, reason in enumerate(reasons):
        if reason in knn_recovery_reasons:
            training_labels = train_df[reason].to_numpy(copy=True)
            training_labels[masked] = (vote_rates[:, j] >= percentage_threshold / 100).astype(int)
            # Each selected reason gets all known and recovered rows, including negatives.
            estimators[j] = LogisticRegression(
                C=C, max_iter=500, solver="liblinear", random_state=random_state
            ).fit(X_train, training_labels.astype(int))
            sources = np.where(known, "observed", "knn_prediction")
            recovery_method = "KNN only"
            lineage[f"{reason}_knn_probability"] = np.nan
            lineage.iloc[masked_positions, lineage.columns.get_loc(f"{reason}_knn_probability")] = vote_rates[:, j]
        else:
            training_labels = reference_augmented_df[reason].to_numpy(copy=True)
            available = np.isin(training_labels, [0, 1])
            sources = np.where(known, "observed", np.where(available, "cluster_prediction", "unresolved"))
            recovery_method = "Reused cluster-only estimator"
            assert estimators[j] is reference_model.estimators_[j]

        available = np.isin(training_labels, [0, 1])
        recovered = masked & available
        lineage[reason] = training_labels
        lineage[f"{reason}_source"] = sources
        coverage_rows.append({
            "Reason": reason,
            "Recovery": recovery_method,
            "Training rows": int(available.sum()),
            "Observed rows": int(known.sum()),
            "Recovered rows": int(recovered.sum()),
            "Recovered positives": int((recovered & (training_labels == 1)).sum()),
            "Recovered negatives": int((recovered & (training_labels == 0)).sum()),
            "Unresolved rows": int((~available).sum()),
            "Percent unresolved": float((~available).mean() * 100),
        })

    return SelectiveReasonClassifier(estimators), pd.DataFrame(coverage_rows).set_index("Reason"), lineage


In [144]:
# Decision2026-11-06: use KNN only for Funding; see EXPERIMENT_LOG.md.
# Safety/Efficacy/Recruitment/Other keep the exact fitted cluster-only estimators.
SELECTIVE_CLUSTER_COUNT = 14
KNN_RECOVERY_REASONS = ["Funding"]
CLUSTER_RECOVERY_REASONS = ["Safety", "Efficacy", "Recruitment", "Other"]
SELECTIVE_NEIGHBORS = 3
SELECTIVE_THRESHOLD = 50

if "pipeline_results" not in globals() or SELECTIVE_CLUSTER_COUNT not in pipeline_results:
    raise ValueError("Run the cluster-count sweep first so pipeline_results contains K=14.")
assert set(KNN_RECOVERY_REASONS).isdisjoint(CLUSTER_RECOVERY_REASONS)
assert set(KNN_RECOVERY_REASONS + CLUSTER_RECOVERY_REASONS) == set([
    "Safety", "Funding", "Efficacy", "Recruitment", "Other"
])
selective_reference_run = pipeline_results[SELECTIVE_CLUSTER_COUNT]
selective_reference_model = selective_reference_run["reason_model"]
selective_reason_model, selective_coverage, selective_label_lineage = fit_selective_reason_model(
    selective_reference_model, X_train, part1_train,
    selective_reference_run["part1_train_augmented"],
    knn_recovery_reasons=KNN_RECOVERY_REASONS,
    n_neighbors=SELECTIVE_NEIGHBORS,
    percentage_threshold=SELECTIVE_THRESHOLD,
    C=1.0, random_state=RANDOM_STATE,
)
selective_validation_metrics = evaluate_reason_model(selective_reason_model, X_val, part1_val)
selective_experiment = {
    "reason_model": selective_reason_model,
    "validation_metrics": selective_validation_metrics,
    "coverage": selective_coverage,
    "label_lineage": selective_label_lineage,
    "settings": {
        "reference_cluster_count": SELECTIVE_CLUSTER_COUNT,
        "knn_recovery_reasons": KNN_RECOVERY_REASONS,
        "cluster_recovery_reasons": CLUSTER_RECOVERY_REASONS,
        "n_neighbors": SELECTIVE_NEIGHBORS,
        "neighbor_space": "original 150-dimensional scaled embeddings",
        "percentage_threshold": SELECTIVE_THRESHOLD,
        "C": 1.0,
        "random_state": RANDOM_STATE,
    },
}


In [145]:
selective_comparison = pd.DataFrame({
    "Cluster only": selective_reference_run["validation_metrics"],
    "Selective KNN recovery": selective_validation_metrics,
}).T
selective_comparison["Baseline sample-wise F1"] = baseline_validation_metrics["Sample-wise F1"]
selective_comparison["Delta vs baseline"] = (
    selective_comparison["Sample-wise F1"] - selective_comparison["Baseline sample-wise F1"]
)
with pd.option_context("display.max_columns", None):
    display(selective_comparison.round(4))
    # Coverage is per reason because the five models now use different training masks.
    display(selective_coverage.round(4))

# Raw protected estimators are identical. The row-level nonempty fallback can still
# change final protected labels when a new Funding prediction affects fallback.
selective_reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
reference_raw = selective_reference_model.predict(X_val)
selective_raw = selective_reason_model.predict(X_val)
reference_final = predict_reason_labels(selective_reference_model, X_val)
selective_final = predict_reason_labels(selective_reason_model, X_val)
protected_prediction_audit = []
for reason in CLUSTER_RECOVERY_REASONS:
    j = selective_reasons.index(reason)
    assert selective_reason_model.estimators_[j] is selective_reference_model.estimators_[j]
    assert np.array_equal(reference_raw[:, j], selective_raw[:, j])
    protected_prediction_audit.append({
        "Reason": reason,
        "Same fitted estimator": True,
        "Raw prediction changes": int((reference_raw[:, j] != selective_raw[:, j]).sum()),
        "Final prediction changes after fallback": int((reference_final[:, j] != selective_final[:, j]).sum()),
    })
display(pd.DataFrame(protected_prediction_audit).set_index("Reason"))
display(selective_label_lineage.head())
# Existing export remains the cluster-only reference; this model is not selected automatically.


,Sample-wise F1,Macro F1,Safety,Funding,Efficacy,Recruitment,Other,Baseline sample-wise F1,Delta vs baseline
Cluster only,0.8949,0.7221,0.7826,0.3333,0.9333,0.9899,0.5714,0.8782,0.0167
Selective KNN recovery,0.8970,0.7355,0.7826,0.4000,0.9333,0.9899,0.5714,0.8782,0.0188


,Recovery,Training rows,Observed rows,Recovered rows,Recovered positives,Recovered negatives,Unresolved rows,Percent unresolved
Reason,,,,,,,,
Safety,Reused cluster-only estimator,651,157,494,0,494,520,44.4065
Funding,KNN only,1171,157,1014,16,998,0,0.0000
Efficacy,Reused cluster-only estimator,651,157,494,16,478,520,44.4065
Recruitment,Reused cluster-only estimator,651,157,494,458,36,520,44.4065
Other,Reused cluster-only estimator,651,157,494,20,474,520,44.4065


,Same fitted estimator,Raw prediction changes,Final prediction changes after fallback
Reason,,,
Safety,True,0,0
Efficacy,True,0,0
Recruitment,True,0,0
Other,True,0,0


,ID,Safety,Safety_source,Funding_knn_probability,Funding,Funding_source,Efficacy,Efficacy_source,Recruitment,Recruitment_source,Other,Other_source
0,TR08860,-100,unresolved,0.0,0,knn_prediction,-100,unresolved,-100,unresolved,-100,unresolved
1,TR03494,-100,unresolved,0.0,0,knn_prediction,-100,unresolved,-100,unresolved,-100,unresolved
2,TR00772,0,cluster_prediction,0.0,0,knn_prediction,0,cluster_prediction,1,cluster_prediction,0,cluster_prediction
3,TR08870,-100,unresolved,0.0,0,knn_prediction,-100,unresolved,-100,unresolved,-100,unresolved
4,TR02827,0,cluster_prediction,0.0,0,knn_prediction,0,cluster_prediction,0,cluster_prediction,1,cluster_prediction


In [146]:
def export_part1_predictions(model, X_test, test_df, output_path):
    """Save Kaggle ID,Predicted rows with at least one reason per trial."""
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    if len(X_test) != len(test_df):
        raise ValueError("Test embeddings and trial rows must have matching lengths and order.")
    predictions = predict_reason_labels(model, X_test)
    if predictions.shape != (len(test_df), len(reason_columns)):
        raise ValueError("Expected one five-label prediction vector per test trial.")
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError("Reason predictions must be binary.")
    if test_df["ID"].isna().any() or not test_df["ID"].is_unique:
        raise ValueError("Test trial IDs must be present and unique.")

    submission = test_df[["ID"]].copy()
    submission["Predicted"] = [
        " ".join(reason for reason, value in zip(reason_columns, row) if value == 1)
        for row in predictions
    ]
    submission.to_csv(output_path, index=False)
    return submission


In [147]:
# This export still uses the cluster-sweep model; regularization models are not selected automatically.
# Choose a run explicitly; classifier inputs are the original scaled embeddings.
# Kernel PCA was used only to recover labels during training.
KMEANS_EXPORT_CLUSTERS = 21
selected_run = pipeline_results[KMEANS_EXPORT_CLUSTERS]
selected_reason_model = selected_run["reason_model"]
submission_path = ATTEMPT_DIR / "part1_kmeans_experiments_submission.csv"
part1_submission = export_part1_predictions(
    selected_reason_model, X_test, part1_test, submission_path
)
print("Saved:", submission_path)
print("Test trials:", len(part1_submission))
print("Trials with no predicted reasons:", part1_submission["Predicted"].eq("").sum())
print("Highest-probability fallback trials:", (~selected_reason_model.predict(X_test).any(axis=1)).sum())
part1_submission.head()


Saved: /Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm/part1_kmeans_experiments_submission.csv
Test trials: 312
Trials with no predicted reasons: 0
Highest-probability fallback trials: 9


,ID,Predicted
0,TR00756,Safety Efficacy
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety


In [148]:
# Export the selected Funding-only KNN hybrid; keep the cluster-only CSV separate.
if "selective_experiment" not in globals():
    raise ValueError("Run the K=14 cluster sweep and Funding-only selective experiment first.")
if selective_experiment["settings"]["knn_recovery_reasons"] != ["Funding"]:
    raise ValueError("This export requires Funding-only KNN recovery.")

funding_hybrid_model = selective_experiment["reason_model"]
funding_hybrid_submission_path = ATTEMPT_DIR / "part1_kmeans_funding_knn_submission.csv"
funding_hybrid_submission = export_part1_predictions(
    funding_hybrid_model, X_test, part1_test, funding_hybrid_submission_path
)
assert funding_hybrid_submission["Predicted"].notna().all()
assert funding_hybrid_submission["Predicted"].str.strip().ne("").all()
print("Saved:", funding_hybrid_submission_path)
print("Test trials:", len(funding_hybrid_submission))
print("Highest-probability fallback trials:", (~funding_hybrid_model.predict(X_test).any(axis=1)).sum())
funding_hybrid_submission.head()


Saved: /Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm/part1_kmeans_funding_knn_submission.csv
Test trials: 312
Highest-probability fallback trials: 10


,ID,Predicted
0,TR00756,Safety Efficacy Other
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety


In [149]:
# Controlled comparison: Funding-only KNN recovery with K=21 clustering.
# Run the shared K=2..30 sweep first; this reuses its K=21 fitted cluster model.
K21_FUNDING_CLUSTER_COUNT = 21
if "pipeline_results" not in globals() or K21_FUNDING_CLUSTER_COUNT not in pipeline_results:
    raise ValueError("Run the cluster-count sweep first so pipeline_results contains K=21.")

k21_funding_reference = pipeline_results[K21_FUNDING_CLUSTER_COUNT]
k21_funding_model, k21_funding_coverage, k21_funding_lineage = fit_selective_reason_model(
    k21_funding_reference["reason_model"], X_train, part1_train,
    k21_funding_reference["part1_train_augmented"],
    knn_recovery_reasons=("Funding",), n_neighbors=3,
    percentage_threshold=50, C=1.0, random_state=RANDOM_STATE,
)
k21_funding_validation_metrics = evaluate_reason_model(k21_funding_model, X_val, part1_val)
k21_funding_submission_path = ATTEMPT_DIR / "part1_kmeans_funding_knn_k21_submission.csv"
k21_funding_submission = export_part1_predictions(
    k21_funding_model, X_test, part1_test, k21_funding_submission_path
)
assert k21_funding_submission["Predicted"].notna().all()
assert k21_funding_submission["Predicted"].str.strip().ne("").all()
print("K=21 Funding-only validation metrics:")
print(k21_funding_validation_metrics.to_string())
print("Per-reason training coverage:")
print(k21_funding_coverage.to_string())
print("Saved:", k21_funding_submission_path)
print("Test trials:", len(k21_funding_submission))
print("Highest-probability fallback trials:", (~k21_funding_model.predict(X_test).any(axis=1)).sum())
k21_funding_submission.head()


K=21 Funding-only validation metrics:
Sample-wise F1    0.890171
Macro F1          0.740560
Safety            0.857143
Funding           0.400000
Efficacy          0.866667
Recruitment       0.970297
Other             0.608696
Per-reason training coverage:
                                  Recovery  Training rows  Observed rows  Recovered rows  Recovered positives  Recovered negatives  Unresolved rows  Percent unresolved
Reason                                                                                                                                                                 
Safety       Reused cluster-only estimator            624            157             467                    0                  467              547           46.712212
Funding                           KNN only           1171            157            1014                   16                  998                0            0.000000
Efficacy     Reused cluster-only estimator            624            15

,ID,Predicted
0,TR00756,Safety Efficacy
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety


In [150]:
def fit_selective_smoothed_blend(reference_run, X_train, X_cluster, train_df,
                                 blend_reasons=("Safety", "Funding", "Other"),
                                 n_neighbors=3, cluster_score_weight=0.5,
                                 smoothing_strength=2.0, percentage_threshold=50,
                                 C=1.0, random_state=42, pseudo_weight=1.0):
    """Blend smoothed K=21 cluster rates and original-label NN votes for selected reasons.

    Only originally observed labels estimate rates and neighbor votes. Selected reasons
    receive binary labels for every masked trial; protected estimators are reused exactly.
    """
    reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    selected = set(blend_reasons)
    if selected != {"Safety", "Funding", "Other"}:
        raise ValueError("This controlled experiment blends Safety, Funding, and Other only.")
    if not (0 <= cluster_score_weight <= 1 and 0 <= percentage_threshold <= 100):
        raise ValueError("Blend weight and recovery threshold are out of range.")
    if smoothing_strength < 0:
        raise ValueError("Smoothing strength must be nonnegative.")
    if pseudo_weight <= 0:
        raise ValueError("Pseudo-label weight must be positive.")
    X_train = np.asarray(X_train)
    X_cluster = np.asarray(X_cluster)
    if len(X_train) != len(train_df) or len(X_cluster) != len(train_df):
        raise ValueError("Embedding rows must match training rows.")
    augmented = reference_run["part1_train_augmented"]
    if not augmented["ID"].reset_index(drop=True).equals(train_df["ID"].reset_index(drop=True)):
        raise ValueError("Reference augmented trial order differs.")
    known = train_df[reasons].isin([0, 1]).all(axis=1).to_numpy()
    masked = train_df[reasons].eq(-100).all(axis=1).to_numpy()
    if not (known | masked).all() or known.sum() != 157:
        raise ValueError("Expected original fully observed or fully masked labels.")
    if not np.array_equal(augmented.loc[known, reasons].to_numpy(), train_df.loc[known, reasons].to_numpy()):
        raise ValueError("Reference data changed original labels.")
    if not 1 <= n_neighbors <= int(known.sum()):
        raise ValueError("Invalid original-label neighbor count.")

    y_known = train_df.loc[known, reasons].to_numpy(dtype=int)
    priors = y_known.mean(axis=0)
    cluster_ids = reference_run["kmeans_model"].predict(X_cluster)
    n_clusters = reference_run["kmeans_model"].n_clusters
    cluster_rates = np.empty((n_clusters, len(reasons)), dtype=float)
    known_cluster_ids = cluster_ids[known]
    for cluster in range(n_clusters):
        in_cluster = known_cluster_ids == cluster
        count = int(in_cluster.sum())
        positives = y_known[in_cluster].sum(axis=0)
        cluster_rates[cluster] = (positives + smoothing_strength * priors) / (count + smoothing_strength)

    neighbors = NearestNeighbors(n_neighbors=n_neighbors).fit(X_train[known])
    masked_positions = np.flatnonzero(masked)
    vote_rates = y_known[neighbors.kneighbors(X_train[masked], return_distance=False)].mean(axis=1)
    smoothed_rates = cluster_rates[cluster_ids[masked]]
    blended_rates = cluster_score_weight * smoothed_rates + (1 - cluster_score_weight) * vote_rates

    reference_model = reference_run["reason_model"]
    estimators = list(reference_model.estimators_)
    lineage = train_df[["ID"]].copy()
    coverage_rows = []
    for j, reason in enumerate(reasons):
        if reason in selected:
            labels = train_df[reason].to_numpy(copy=True)
            labels[masked] = (blended_rates[:, j] >= percentage_threshold / 100).astype(int)
            estimators[j] = LogisticRegression(
                C=C, max_iter=500, solver="liblinear", random_state=random_state,
            ).fit(X_train, labels.astype(int), sample_weight=np.where(known, 1.0, pseudo_weight))
            source = np.where(known, "observed", "smoothed_cluster_knn_blend")
            lineage[f"{reason}_cluster_rate"] = np.nan
            lineage[f"{reason}_neighbor_vote"] = np.nan
            lineage[f"{reason}_blended_rate"] = np.nan
            lineage.loc[masked, f"{reason}_cluster_rate"] = smoothed_rates[:, j]
            lineage.loc[masked, f"{reason}_neighbor_vote"] = vote_rates[:, j]
            lineage.loc[masked, f"{reason}_blended_rate"] = blended_rates[:, j]
        else:
            labels = augmented[reason].to_numpy(copy=True)
            available = np.isin(labels, [0, 1])
            source = np.where(known, "observed", np.where(available, "cluster_prediction", "unresolved"))
            assert estimators[j] is reference_model.estimators_[j]
        available = np.isin(labels, [0, 1])
        recovered = masked & available
        lineage[reason] = labels
        lineage[f"{reason}_source"] = source
        coverage_rows.append({
            "Reason": reason,
            "Training rows": int(available.sum()),
            "Observed rows": int(known.sum()),
            "Recovered rows": int(recovered.sum()),
            "Recovered positives": int((recovered & (labels == 1)).sum()),
            "Recovered negatives": int((recovered & (labels == 0)).sum()),
            "Unresolved rows": int((~available).sum()),
            "Percent unresolved": float((~available).mean() * 100),
        })
    return SelectiveReasonClassifier(estimators), pd.DataFrame(coverage_rows).set_index("Reason"), lineage


In [151]:
# K21 blend with protected final Efficacy/Recruitment predictions.
class ProtectedReasonFallbackClassifier:
    """Preserve protected K21 decisions, filling empty rows from selected reasons."""
    def __init__(self, selective_model, reference_model,
                 selected_reasons=("Safety", "Funding", "Other")):
        self.selective_model = selective_model
        self.reference_model = reference_model
        self.reason_order = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
        self.selected_indices = np.array([self.reason_order.index(r) for r in selected_reasons])
        self.protected_indices = np.array([j for j in range(5) if j not in self.selected_indices])
        self.estimators_ = selective_model.estimators_

    def _predict_and_count(self, X):
        final = np.asarray(self.selective_model.predict(X), dtype=int).copy()
        reference_final = predict_reason_labels(self.reference_model, X)
        final[:, self.protected_indices] = reference_final[:, self.protected_indices]
        empty_positions = np.flatnonzero(~final.any(axis=1))
        if len(empty_positions):
            selected_probabilities = self.selective_model.predict_proba(X[empty_positions])[:, self.selected_indices]
            selected_winners = self.selected_indices[selected_probabilities.argmax(axis=1)]
            final[empty_positions, selected_winners] = 1
        return final, len(empty_positions)

    def predict(self, X):
        return self._predict_and_count(X)[0]

    def predict_proba(self, X):
        return self.selective_model.predict_proba(X)

    def selected_fallback_count(self, X):
        return self._predict_and_count(X)[1]


BLEND_K = 21
if "pipeline_results" not in globals() or BLEND_K not in pipeline_results:
    raise ValueError("Run the cluster-count sweep first so pipeline_results contains K=21.")
blend_reference = pipeline_results[BLEND_K]
blend_reason_model, blend_coverage, blend_lineage = fit_selective_smoothed_blend(
    blend_reference, X_train, X_train_cluster, part1_train,
    blend_reasons=("Safety", "Funding", "Other"),
    n_neighbors=3, cluster_score_weight=0.5,
    smoothing_strength=2.0, percentage_threshold=50,
    C=1.0, random_state=RANDOM_STATE,
)
blend_prediction_model = ProtectedReasonFallbackClassifier(
    blend_reason_model, blend_reference["reason_model"]
)
blend_validation_metrics = evaluate_reason_model(blend_prediction_model, X_val, part1_val)
protected_reasons = ["Efficacy", "Recruitment"]
reason_order = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
reference_val_final = predict_reason_labels(blend_reference["reason_model"], X_val)
blend_val_final = predict_reason_labels(blend_prediction_model, X_val)
reference_test_final = predict_reason_labels(blend_reference["reason_model"], X_test)
blend_test_final = predict_reason_labels(blend_prediction_model, X_test)
reference_raw = blend_reference["reason_model"].predict(X_val)
blend_raw = blend_reason_model.predict(X_val)
blend_audit = pd.DataFrame({
    "Reason": protected_reasons,
    "Estimator reused": [blend_reason_model.estimators_[reason_order.index(r)] is blend_reference["reason_model"].estimators_[reason_order.index(r)] for r in protected_reasons],
    "Raw validation changes": [int((reference_raw[:, reason_order.index(r)] != blend_raw[:, reason_order.index(r)]).sum()) for r in protected_reasons],
    "Final validation changes": [int((reference_val_final[:, reason_order.index(r)] != blend_val_final[:, reason_order.index(r)]).sum()) for r in protected_reasons],
    "Final test changes": [int((reference_test_final[:, reason_order.index(r)] != blend_test_final[:, reason_order.index(r)]).sum()) for r in protected_reasons],
})
assert blend_audit["Estimator reused"].all()
assert blend_audit[["Raw validation changes", "Final validation changes", "Final test changes"]].eq(0).all().all()
assert blend_val_final.any(axis=1).all() and blend_test_final.any(axis=1).all()

blend_submission_path = ATTEMPT_DIR / "part1_kmeans_blend_safety_funding_other_k21_submission.csv"
blend_submission = export_part1_predictions(
    blend_prediction_model, X_test, part1_test, blend_submission_path
)
assert blend_submission["Predicted"].notna().all()
assert blend_submission["Predicted"].str.strip().ne("").all()
print("K21 cluster-only vs corrected selective blend validation:")
print(pd.DataFrame({"Cluster only": blend_reference["validation_metrics"], "Selective blend": blend_validation_metrics}).to_string())
print("Per-reason training coverage:")
print(blend_coverage.to_string())
print("Protected final decision audit:")
print(blend_audit.to_string(index=False))
print("Saved:", blend_submission_path)
print("Test trials:", len(blend_submission))
print("Selected-reason fallback trials:", blend_prediction_model.selected_fallback_count(X_test))
blend_submission.head()


K21 cluster-only vs corrected selective blend validation:
                Cluster only  Selective blend
Sample-wise F1      0.888034         0.850427
Macro F1            0.727227         0.600726
Safety              0.857143         0.500000
Funding             0.333333         0.333333
Efficacy            0.866667         0.866667
Recruitment         0.970297         0.970297
Other               0.608696         0.333333
Per-reason training coverage:
             Training rows  Observed rows  Recovered rows  Recovered positives  Recovered negatives  Unresolved rows  Percent unresolved
Reason                                                                                                                                  
Safety                1171            157            1014                   44                  970                0            0.000000
Funding               1171            157            1014                    5                 1009                0            0.000

,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Funding


In [152]:
# Controlled variant: quarter-weight pseudo labels for Safety, Funding, Other only.
# Run the shared K=21 sweep and the corrected blend cell above first.
WEIGHTED_BLEND_K = 21
PSEUDO_WEIGHT = 0.25
if "pipeline_results" not in globals() or WEIGHTED_BLEND_K not in pipeline_results:
    raise ValueError("Run the cluster-count sweep first so pipeline_results contains K=21.")
if "ProtectedReasonFallbackClassifier" not in globals():
    raise ValueError("Run the corrected blend cell first to define its prediction adapter.")
weighted_blend_reference = pipeline_results[WEIGHTED_BLEND_K]
weighted_blend_reason_model, weighted_blend_coverage, weighted_blend_lineage = fit_selective_smoothed_blend(
    weighted_blend_reference, X_train, X_train_cluster, part1_train,
    blend_reasons=("Safety", "Funding", "Other"),
    n_neighbors=3, cluster_score_weight=0.5,
    smoothing_strength=2.0, percentage_threshold=50,
    C=1.0, random_state=RANDOM_STATE, pseudo_weight=PSEUDO_WEIGHT,
)
weighted_blend_prediction_model = ProtectedReasonFallbackClassifier(
    weighted_blend_reason_model, weighted_blend_reference["reason_model"]
)
weighted_blend_validation_metrics = evaluate_reason_model(
    weighted_blend_prediction_model, X_val, part1_val
)
weighted_reference_final_val = predict_reason_labels(weighted_blend_reference["reason_model"], X_val)
weighted_reference_final_test = predict_reason_labels(weighted_blend_reference["reason_model"], X_test)
weighted_final_val = predict_reason_labels(weighted_blend_prediction_model, X_val)
weighted_final_test = predict_reason_labels(weighted_blend_prediction_model, X_test)
weighted_blend_audit = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "Estimator reused": [weighted_blend_reason_model.estimators_[j] is weighted_blend_reference["reason_model"].estimators_[j] for j in [2, 3]],
    "Raw validation changes": [int((weighted_blend_reason_model.predict(X_val)[:, j] != weighted_blend_reference["reason_model"].predict(X_val)[:, j]).sum()) for j in [2, 3]],
    "Final validation changes": [int((weighted_final_val[:, j] != weighted_reference_final_val[:, j]).sum()) for j in [2, 3]],
    "Final test changes": [int((weighted_final_test[:, j] != weighted_reference_final_test[:, j]).sum()) for j in [2, 3]],
})
assert weighted_blend_audit["Estimator reused"].all()
assert weighted_blend_audit[["Raw validation changes", "Final validation changes", "Final test changes"]].eq(0).all().all()
assert weighted_final_val.any(axis=1).all() and weighted_final_test.any(axis=1).all()
weighted_blend_submission_path = ATTEMPT_DIR / "part1_kmeans_blend_sfo_k21_weight025_submission.csv"
weighted_blend_submission = export_part1_predictions(
    weighted_blend_prediction_model, X_test, part1_test, weighted_blend_submission_path
)
assert weighted_blend_submission["Predicted"].notna().all()
assert weighted_blend_submission["Predicted"].str.strip().ne("").all()
print("K21 unweighted vs quarter-weight selective blend validation:")
print(pd.DataFrame({"Unweighted blend": blend_validation_metrics, "Quarter-weight blend": weighted_blend_validation_metrics}).to_string())
print("Per-reason training coverage:")
print(weighted_blend_coverage.to_string())
print("Protected final decision audit:")
print(weighted_blend_audit.to_string(index=False))
print("Saved:", weighted_blend_submission_path)
print("Test trials:", len(weighted_blend_submission))
print("Selected-reason fallback trials:", weighted_blend_prediction_model.selected_fallback_count(X_test))
weighted_blend_submission.head()


K21 unweighted vs quarter-weight selective blend validation:
                Unweighted blend  Quarter-weight blend
Sample-wise F1          0.850427              0.880342
Macro F1                0.600726              0.704059
Safety                  0.500000              0.750000
Funding                 0.333333              0.400000
Efficacy                0.866667              0.866667
Recruitment             0.970297              0.970297
Other                   0.333333              0.533333
Per-reason training coverage:
             Training rows  Observed rows  Recovered rows  Recovered positives  Recovered negatives  Unresolved rows  Percent unresolved
Reason                                                                                                                                  
Safety                1171            157            1014                   44                  970                0            0.000000
Funding               1171            157            1014 

,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety


In [153]:
# Controlled variant: lower selected-reason recovery threshold to 30%.
# Keep quarter-weight pseudo labels, K21, smoothing2, 50/50 scores, and protected E/R decisions.
THRESHOLD30_K = 21
if "pipeline_results" not in globals() or THRESHOLD30_K not in pipeline_results:
    raise ValueError("Run the K=21 cluster sweep first.")
if "ProtectedReasonFallbackClassifier" not in globals():
    raise ValueError("Run the corrected blend cell first to define its prediction adapter.")
threshold30_reference = pipeline_results[THRESHOLD30_K]
threshold30_reason_model, threshold30_coverage, threshold30_lineage = fit_selective_smoothed_blend(
    threshold30_reference, X_train, X_train_cluster, part1_train,
    blend_reasons=("Safety", "Funding", "Other"),
    n_neighbors=3, cluster_score_weight=0.5,
    smoothing_strength=2.0, percentage_threshold=30,
    C=1.0, random_state=RANDOM_STATE, pseudo_weight=0.25,
)
threshold30_prediction_model = ProtectedReasonFallbackClassifier(
    threshold30_reason_model, threshold30_reference["reason_model"]
)
threshold30_validation_metrics = evaluate_reason_model(
    threshold30_prediction_model, X_val, part1_val
)
reason_order = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
originally_known = part1_train[reason_order].isin([0, 1]).all(axis=1).to_numpy()
assert np.array_equal(
    threshold30_lineage.loc[originally_known, reason_order].to_numpy(),
    part1_train.loc[originally_known, reason_order].to_numpy(),
)
reference_final_val = predict_reason_labels(threshold30_reference["reason_model"], X_val)
reference_final_test = predict_reason_labels(threshold30_reference["reason_model"], X_test)
threshold30_final_val = predict_reason_labels(threshold30_prediction_model, X_val)
threshold30_final_test = predict_reason_labels(threshold30_prediction_model, X_test)
threshold30_audit = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "Estimator reused": [threshold30_reason_model.estimators_[j] is threshold30_reference["reason_model"].estimators_[j] for j in [2, 3]],
    "Final validation changes": [int((threshold30_final_val[:, j] != reference_final_val[:, j]).sum()) for j in [2, 3]],
    "Final test changes": [int((threshold30_final_test[:, j] != reference_final_test[:, j]).sum()) for j in [2, 3]],
})
assert threshold30_audit["Estimator reused"].all()
assert threshold30_audit[["Final validation changes", "Final test changes"]].eq(0).all().all()
assert threshold30_final_val.any(axis=1).all() and threshold30_final_test.any(axis=1).all()
threshold30_submission_path = ATTEMPT_DIR / "part1_kmeans_blend_sfo_k21_weight025_threshold30_submission.csv"
threshold30_submission = export_part1_predictions(
    threshold30_prediction_model, X_test, part1_test, threshold30_submission_path
)
assert threshold30_submission["Predicted"].notna().all()
assert threshold30_submission["Predicted"].str.strip().ne("").all()
print("Quarter-weight blend: threshold 50 vs 30 validation:")
if "weighted_blend_validation_metrics" in globals():
    print(pd.DataFrame({"Threshold 50": weighted_blend_validation_metrics, "Threshold 30": threshold30_validation_metrics}).to_string())
else:
    print(threshold30_validation_metrics.to_string())
print("Per-reason coverage:")
print(threshold30_coverage.to_string())
print("Protected final decision audit:")
print(threshold30_audit.to_string(index=False))
print("Saved:", threshold30_submission_path)
print("Test trials:", len(threshold30_submission))
print("Selected-reason fallback trials:", threshold30_prediction_model.selected_fallback_count(X_test))
threshold30_submission.head()


Quarter-weight blend: threshold 50 vs 30 validation:
                Threshold 50  Threshold 30
Sample-wise F1      0.880342      0.893162
Macro F1            0.704059      0.775236
Safety              0.750000      0.705882
Funding             0.400000      0.666667
Efficacy            0.866667      0.866667
Recruitment         0.970297      0.970297
Other               0.533333      0.666667
Per-reason coverage:
             Training rows  Observed rows  Recovered rows  Recovered positives  Recovered negatives  Unresolved rows  Percent unresolved
Reason                                                                                                                                  
Safety                1171            157            1014                   75                  939                0            0.000000
Funding               1171            157            1014                   16                  998                0            0.000000
Efficacy               624        

,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety


In [154]:
# Controlled regularization test: C=0.1 for selected Safety/Funding/Other only.
# Recovery and pseudo-label weights match the threshold-30 C=1 blend above.
C01_BLEND_K = 21
if "pipeline_results" not in globals() or C01_BLEND_K not in pipeline_results:
    raise ValueError("Run the K=21 cluster sweep first.")
if "ProtectedReasonFallbackClassifier" not in globals():
    raise ValueError("Run the corrected blend cell first to define its prediction adapter.")
c01_reference = pipeline_results[C01_BLEND_K]
c01_reason_model, c01_coverage, c01_lineage = fit_selective_smoothed_blend(
    c01_reference, X_train, X_train_cluster, part1_train,
    blend_reasons=("Safety", "Funding", "Other"),
    n_neighbors=3, cluster_score_weight=0.5,
    smoothing_strength=2.0, percentage_threshold=30,
    C=0.1, random_state=RANDOM_STATE, pseudo_weight=0.25,
)
c01_prediction_model = ProtectedReasonFallbackClassifier(
    c01_reason_model, c01_reference["reason_model"]
)
c01_validation_metrics = evaluate_reason_model(c01_prediction_model, X_val, part1_val)
reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
originally_known = part1_train[reasons].isin([0, 1]).all(axis=1).to_numpy()
assert np.array_equal(c01_lineage.loc[originally_known, reasons].to_numpy(), part1_train.loc[originally_known, reasons].to_numpy())
assert np.array_equal(c01_lineage.loc[:, reasons].to_numpy(), threshold30_lineage.loc[:, reasons].to_numpy())
c01_reference_final_val = predict_reason_labels(c01_reference["reason_model"], X_val)
c01_reference_final_test = predict_reason_labels(c01_reference["reason_model"], X_test)
c01_final_val = predict_reason_labels(c01_prediction_model, X_val)
c01_final_test = predict_reason_labels(c01_prediction_model, X_test)
c01_audit = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "Estimator reused": [c01_reason_model.estimators_[j] is c01_reference["reason_model"].estimators_[j] for j in [2, 3]],
    "Final validation changes": [int((c01_final_val[:, j] != c01_reference_final_val[:, j]).sum()) for j in [2, 3]],
    "Final test changes": [int((c01_final_test[:, j] != c01_reference_final_test[:, j]).sum()) for j in [2, 3]],
})
assert c01_audit["Estimator reused"].all()
assert c01_audit[["Final validation changes", "Final test changes"]].eq(0).all().all()
assert c01_final_val.any(axis=1).all() and c01_final_test.any(axis=1).all()
c01_submission_path = ATTEMPT_DIR / "part1_kmeans_blend_sfo_k21_weight025_threshold30_c01_submission.csv"
c01_submission = export_part1_predictions(c01_prediction_model, X_test, part1_test, c01_submission_path)
assert c01_submission["Predicted"].notna().all() and c01_submission["Predicted"].str.strip().ne("").all()
print("K21 threshold30 quarter-weight C comparison:")
print(pd.DataFrame({"C=1": threshold30_validation_metrics, "C=0.1": c01_validation_metrics}).to_string())
print("Per-reason training coverage:")
print(c01_coverage.to_string())
print("Protected final decision audit:")
print(c01_audit.to_string(index=False))
print("Saved:", c01_submission_path)
print("Test trials:", len(c01_submission))
print("Selected-reason fallback trials:", c01_prediction_model.selected_fallback_count(X_test))
c01_submission.head()


K21 threshold30 quarter-weight C comparison:
                     C=1     C=0.1
Sample-wise F1  0.893162  0.914530
Macro F1        0.775236  0.822504
Safety          0.705882  0.888889
Funding         0.666667  0.666667
Efficacy        0.866667  0.866667
Recruitment     0.970297  0.970297
Other           0.666667  0.720000
Per-reason training coverage:
             Training rows  Observed rows  Recovered rows  Recovered positives  Recovered negatives  Unresolved rows  Percent unresolved
Reason                                                                                                                                  
Safety                1171            157            1014                   75                  939                0            0.000000
Funding               1171            157            1014                   16                  998                0            0.000000
Efficacy               624            157             467                   16                  4

,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety


In [ ]:
# K14 version of the current public-best K21 C1 selective blend.
# Same recovery/weighting settings; protected E/R predictions follow K14 cluster-only.
K14_BLEND_K = 14
if "pipeline_results" not in globals() or K14_BLEND_K not in pipeline_results:
    raise ValueError("Run the cluster-count sweep first so pipeline_results contains K=14.")
if "ProtectedReasonFallbackClassifier" not in globals():
    raise ValueError("Run the corrected blend cell first to define its prediction adapter.")
k14_blend_reference = pipeline_results[K14_BLEND_K]
k14_blend_reason_model, k14_blend_coverage, k14_blend_lineage = fit_selective_smoothed_blend(
    k14_blend_reference, X_train, X_train_cluster, part1_train,
    blend_reasons=("Safety", "Funding", "Other"),
    n_neighbors=3, cluster_score_weight=0.5,
    smoothing_strength=2.0, percentage_threshold=30,
    C=1.0, random_state=RANDOM_STATE, pseudo_weight=0.25,
)
k14_blend_prediction_model = ProtectedReasonFallbackClassifier(
    k14_blend_reason_model, k14_blend_reference["reason_model"]
)
k14_blend_validation_metrics = evaluate_reason_model(
    k14_blend_prediction_model, X_val, part1_val
)
reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
originally_known = part1_train[reasons].isin([0, 1]).all(axis=1).to_numpy()
assert np.array_equal(k14_blend_lineage.loc[originally_known, reasons].to_numpy(), part1_train.loc[originally_known, reasons].to_numpy())
k14_reference_final_val = predict_reason_labels(k14_blend_reference["reason_model"], X_val)
k14_reference_final_test = predict_reason_labels(k14_blend_reference["reason_model"], X_test)
k14_final_val = predict_reason_labels(k14_blend_prediction_model, X_val)
k14_final_test = predict_reason_labels(k14_blend_prediction_model, X_test)
k14_blend_audit = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "Estimator reused": [k14_blend_reason_model.estimators_[j] is k14_blend_reference["reason_model"].estimators_[j] for j in [2, 3]],
    "Final validation changes": [int((k14_final_val[:, j] != k14_reference_final_val[:, j]).sum()) for j in [2, 3]],
    "Final test changes": [int((k14_final_test[:, j] != k14_reference_final_test[:, j]).sum()) for j in [2, 3]],
})
assert k14_blend_audit["Estimator reused"].all()
assert k14_blend_audit[["Final validation changes", "Final test changes"]].eq(0).all().all()
assert k14_final_val.any(axis=1).all() and k14_final_test.any(axis=1).all()
k14_blend_submission_path = ATTEMPT_DIR / "part1_kmeans_blend_sfo_k14_weight025_threshold30_c1_submission.csv"
k14_blend_submission = export_part1_predictions(
    k14_blend_prediction_model, X_test, part1_test, k14_blend_submission_path
)
assert k14_blend_submission["Predicted"].notna().all() and k14_blend_submission["Predicted"].str.strip().ne("").all()
print("K14 selective blend validation:")
print(k14_blend_validation_metrics.to_string())
print("Per-reason training coverage:")
print(k14_blend_coverage.to_string())
print("Protected final decision audit:")
print(k14_blend_audit.to_string(index=False))
print("Saved:", k14_blend_submission_path)
print("Test trials:", len(k14_blend_submission))
print("Selected-reason fallback trials:", k14_blend_prediction_model.selected_fallback_count(X_test))
k14_blend_submission.head()


In [ ]:
# Controlled K14 change: stronger regularization for Safety alone.
# Reuse all recovered labels and four exact fitted C1 estimators.
class SafetyOnlyFallbackClassifier:
    """Keep four final K14 blend decisions fixed; fill empty rows with Safety."""
    def __init__(self, safety_variant_model, reference_prediction_model):
        self.safety_variant_model = safety_variant_model
        self.reference_prediction_model = reference_prediction_model
        self.estimators_ = safety_variant_model.estimators_

    def predict(self, X):
        final = predict_reason_labels(self.reference_prediction_model, X).copy()
        final[:, 0] = self.safety_variant_model.estimators_[0].predict(X)
        final[~final.any(axis=1), 0] = 1
        return final

    def predict_proba(self, X):
        return self.safety_variant_model.predict_proba(X)

    def safety_fallback_count(self, X):
        final = predict_reason_labels(self.reference_prediction_model, X).copy()
        final[:, 0] = self.safety_variant_model.estimators_[0].predict(X)
        return int((~final.any(axis=1)).sum())


if "k14_blend_reason_model" not in globals() or "k14_blend_lineage" not in globals():
    raise ValueError("Run the K14 C1 blend cell first.")
reason_order = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
known_mask = part1_train[reason_order].isin([0, 1]).all(axis=1).to_numpy()
masked_mask = part1_train[reason_order].eq(-100).all(axis=1).to_numpy()
assert (known_mask | masked_mask).all()
safety_labels = k14_blend_lineage["Safety"].to_numpy(dtype=int)
assert np.isin(safety_labels, [0, 1]).all()
assert np.array_equal(safety_labels[known_mask], part1_train.loc[known_mask, "Safety"].to_numpy(dtype=int))
safety_sample_weight = np.where(known_mask, 1.0, 0.25)
safety_c01_estimator = LogisticRegression(
    C=0.1, max_iter=500, solver="liblinear", random_state=RANDOM_STATE,
).fit(X_train, safety_labels, sample_weight=safety_sample_weight)
safety_c01_estimators = list(k14_blend_reason_model.estimators_)
safety_c01_estimators[0] = safety_c01_estimator
safety_c01_reason_model = SelectiveReasonClassifier(safety_c01_estimators)
safety_c01_prediction_model = SafetyOnlyFallbackClassifier(
    safety_c01_reason_model, k14_blend_prediction_model
)
safety_c01_validation_metrics = evaluate_reason_model(
    safety_c01_prediction_model, X_val, part1_val
)
reference_val_raw = k14_blend_reason_model.predict(X_val)
variant_val_raw = safety_c01_reason_model.predict(X_val)
reference_val_final = predict_reason_labels(k14_blend_prediction_model, X_val)
variant_val_final = predict_reason_labels(safety_c01_prediction_model, X_val)
reference_test_final = predict_reason_labels(k14_blend_prediction_model, X_test)
variant_test_final = predict_reason_labels(safety_c01_prediction_model, X_test)
safety_c01_audit = pd.DataFrame({
    "Reason": reason_order[1:],
    "Estimator reused": [safety_c01_reason_model.estimators_[j] is k14_blend_reason_model.estimators_[j] for j in range(1, 5)],
    "Raw validation changes": [int((reference_val_raw[:, j] != variant_val_raw[:, j]).sum()) for j in range(1, 5)],
    "Final validation changes": [int((reference_val_final[:, j] != variant_val_final[:, j]).sum()) for j in range(1, 5)],
    "Final test changes": [int((reference_test_final[:, j] != variant_test_final[:, j]).sum()) for j in range(1, 5)],
})
assert safety_c01_audit["Estimator reused"].all()
assert safety_c01_audit[["Raw validation changes", "Final validation changes", "Final test changes"]].eq(0).all().all()
assert variant_val_final.any(axis=1).all() and variant_test_final.any(axis=1).all()
print("K14 C1 blend vs Safety-only C0.1 validation:")
print(pd.DataFrame({"C1 all selected": k14_blend_validation_metrics, "Safety C0.1 only": safety_c01_validation_metrics}).to_string())
print("Unchanged-reason audit:")
print(safety_c01_audit.to_string(index=False))
print("Changed Safety final predictions on validation:", int((reference_val_final[:, 0] != variant_val_final[:, 0]).sum()))
print("Changed Safety final predictions on test:", int((reference_test_final[:, 0] != variant_test_final[:, 0]).sum()))
print("Safety-only fallback trials on validation/test:", safety_c01_prediction_model.safety_fallback_count(X_val), safety_c01_prediction_model.safety_fallback_count(X_test))


In [ ]:
# Separate change after Safety-only C=.1: fill unresolved E/R cluster labels by cluster argmax.
# No other fitted reason model or final S/F/O prediction is changed.
class PreserveSFOFallbackClassifier:
    """Keep reference S/F/O final decisions; resolve empty rows via E/R probabilities."""
    def __init__(self, variant_model, reference_prediction_model):
        self.variant_model = variant_model
        self.reference_prediction_model = reference_prediction_model
        self.estimators_ = variant_model.estimators_

    def _predict_and_count(self, X):
        final = np.asarray(self.variant_model.predict(X), dtype=int).copy()
        reference_final = predict_reason_labels(self.reference_prediction_model, X)
        final[:, [0, 1, 4]] = reference_final[:, [0, 1, 4]]
        empty_positions = np.flatnonzero(~final.any(axis=1))
        if len(empty_positions):
            probabilities = self.variant_model.predict_proba(X[empty_positions])[:, [2, 3]]
            final[empty_positions, np.array([2, 3])[probabilities.argmax(axis=1)]] = 1
        return final, len(empty_positions)

    def predict(self, X):
        return self._predict_and_count(X)[0]

    def predict_proba(self, X):
        return self.variant_model.predict_proba(X)

    def er_fallback_count(self, X):
        return self._predict_and_count(X)[1]


if "safety_c01_reason_model" not in globals() or "k14_blend_reference" not in globals():
    raise ValueError("Run the K14 C1 blend and Safety-only C=.1 cells first.")
reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
known = part1_train[reasons].isin([0, 1]).all(axis=1).to_numpy()
masked = part1_train[reasons].eq(-100).all(axis=1).to_numpy()
assert (known | masked).all()
augmented = k14_blend_reference["part1_train_augmented"]
assert np.array_equal(augmented.loc[known, reasons].to_numpy(), part1_train.loc[known, reasons].to_numpy())
er_unresolved = augmented["Efficacy"].eq(-100).to_numpy() & augmented["Recruitment"].eq(-100).to_numpy()
assert np.array_equal(er_unresolved, augmented["Efficacy"].eq(-100).to_numpy())
assert np.array_equal(er_unresolved, augmented["Recruitment"].eq(-100).to_numpy())
assert er_unresolved.sum() == 520

cluster_ids = k14_blend_reference["kmeans_model"].predict(X_train_cluster)
y_known = part1_train.loc[known, reasons].to_numpy(dtype=int)
known_clusters = cluster_ids[known]
global_priors = y_known.mean(axis=0)
cluster_reason_rates = np.empty((k14_blend_reference["kmeans_model"].n_clusters, len(reasons)))
cluster_known_counts = np.zeros(len(cluster_reason_rates), dtype=int)
for cluster in range(len(cluster_reason_rates)):
    members = known_clusters == cluster
    cluster_known_counts[cluster] = int(members.sum())
    cluster_reason_rates[cluster] = y_known[members].mean(axis=0) if members.any() else global_priors

fallback_positions = np.flatnonzero(er_unresolved)
fallback_rates = cluster_reason_rates[cluster_ids[er_unresolved]]
fallback_reason_indices = fallback_rates.argmax(axis=1)  # Fixed five-reason order breaks ties.
fallback_reasons = np.asarray(reasons)[fallback_reason_indices]
er_fallback_labels = augmented[["Efficacy", "Recruitment"]].copy()
er_fallback_labels.loc[er_unresolved, "Efficacy"] = (fallback_reason_indices == 2).astype(int)
er_fallback_labels.loc[er_unresolved, "Recruitment"] = (fallback_reason_indices == 3).astype(int)
assert er_fallback_labels.isin([0, 1]).all().all()
assert np.array_equal(er_fallback_labels.loc[~er_unresolved].to_numpy(), augmented.loc[~er_unresolved, ["Efficacy", "Recruitment"]].to_numpy())
assert np.array_equal(er_fallback_labels.loc[known].to_numpy(), part1_train.loc[known, ["Efficacy", "Recruitment"]].to_numpy())

er_fallback_estimators = list(safety_c01_reason_model.estimators_)
for j, reason in [(2, "Efficacy"), (3, "Recruitment")]:
    er_fallback_estimators[j] = LogisticRegression(
        C=1.0, max_iter=500, solver="liblinear", random_state=RANDOM_STATE,
    ).fit(X_train, er_fallback_labels[reason].to_numpy(dtype=int))
er_fallback_reason_model = SelectiveReasonClassifier(er_fallback_estimators)
er_fallback_prediction_model = PreserveSFOFallbackClassifier(
    er_fallback_reason_model, safety_c01_prediction_model
)
er_fallback_validation_metrics = evaluate_reason_model(
    er_fallback_prediction_model, X_val, part1_val
)
reference_val_raw = safety_c01_reason_model.predict(X_val)
variant_val_raw = er_fallback_reason_model.predict(X_val)
reference_val_final = predict_reason_labels(safety_c01_prediction_model, X_val)
variant_val_final = predict_reason_labels(er_fallback_prediction_model, X_val)
reference_test_final = predict_reason_labels(safety_c01_prediction_model, X_test)
variant_test_final = predict_reason_labels(er_fallback_prediction_model, X_test)
fixed_reason_indices = [0, 1, 4]
er_fallback_audit = pd.DataFrame({
    "Reason": [reasons[j] for j in fixed_reason_indices],
    "Estimator reused": [er_fallback_reason_model.estimators_[j] is safety_c01_reason_model.estimators_[j] for j in fixed_reason_indices],
    "Raw validation changes": [int((reference_val_raw[:, j] != variant_val_raw[:, j]).sum()) for j in fixed_reason_indices],
    "Final validation changes": [int((reference_val_final[:, j] != variant_val_final[:, j]).sum()) for j in fixed_reason_indices],
    "Final test changes": [int((reference_test_final[:, j] != variant_test_final[:, j]).sum()) for j in fixed_reason_indices],
})
assert er_fallback_audit["Estimator reused"].all()
assert er_fallback_audit[["Raw validation changes", "Final validation changes", "Final test changes"]].eq(0).all().all()
assert variant_val_final.any(axis=1).all() and variant_test_final.any(axis=1).all()
er_fallback_distribution = pd.Series(fallback_reasons).value_counts().reindex(reasons, fill_value=0)
er_fallback_coverage = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "Training rows": [1171, 1171],
    "Observed rows": [int(known.sum())] * 2,
    "Earlier cluster-recovered rows": [int((masked & ~er_unresolved).sum())] * 2,
    "Argmax fallback rows": [int(er_unresolved.sum())] * 2,
    "Final positive labels": [int(er_fallback_labels[r].sum()) for r in ["Efficacy", "Recruitment"]],
    "Final negative labels": [int((er_fallback_labels[r] == 0).sum()) for r in ["Efficacy", "Recruitment"]],
}).set_index("Reason")
er_fallback_confusion = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "TP": [int(((part1_val[r].to_numpy() == 1) & (variant_val_final[:, j] == 1)).sum()) for j, r in [(2, "Efficacy"), (3, "Recruitment")]],
    "FP": [int(((part1_val[r].to_numpy() == 0) & (variant_val_final[:, j] == 1)).sum()) for j, r in [(2, "Efficacy"), (3, "Recruitment")]],
    "FN": [int(((part1_val[r].to_numpy() == 1) & (variant_val_final[:, j] == 0)).sum()) for j, r in [(2, "Efficacy"), (3, "Recruitment")]],
}).set_index("Reason")
print("Safety C=.1 reference vs E/R cluster argmax fallback:")
print(pd.DataFrame({"Safety C=.1 reference": safety_c01_validation_metrics, "E/R fallback": er_fallback_validation_metrics}).to_string())
print("Fallback reason distribution:")
print(er_fallback_distribution.to_string())
print("E/R training coverage:")
print(er_fallback_coverage.to_string())
print("Fixed S/F/O audit:")
print(er_fallback_audit.to_string(index=False))
print("E/R validation confusion:")
print(er_fallback_confusion.to_string())
print("Zero-known-member clusters:", int((cluster_known_counts == 0).sum()))
print("E/R-only nonempty fallback trials on validation/test:", er_fallback_prediction_model.er_fallback_count(X_val), er_fallback_prediction_model.er_fallback_count(X_test))


In [ ]:
# Separate E/R fallback experiment: three original-label neighbors for 520 unresolved rows.
# Keep the K14 Safety C=.1 model and all S/F/O final decisions fixed.
if "safety_c01_reason_model" not in globals() or "k14_blend_reference" not in globals():
    raise ValueError("Run the K14 C1 blend and Safety-only C=.1 cells first.")
if "PreserveSFOFallbackClassifier" not in globals():
    raise ValueError("Run the previous E/R fallback cell to define its prediction adapter.")
reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
known = part1_train[reasons].isin([0, 1]).all(axis=1).to_numpy()
masked = part1_train[reasons].eq(-100).all(axis=1).to_numpy()
assert (known | masked).all()
augmented = k14_blend_reference["part1_train_augmented"]
assert np.array_equal(augmented.loc[known, reasons].to_numpy(), part1_train.loc[known, reasons].to_numpy())
er_unresolved = augmented["Efficacy"].eq(-100).to_numpy() & augmented["Recruitment"].eq(-100).to_numpy()
assert np.array_equal(er_unresolved, augmented["Efficacy"].eq(-100).to_numpy())
assert np.array_equal(er_unresolved, augmented["Recruitment"].eq(-100).to_numpy())
assert int(er_unresolved.sum()) == 520

original_er_labels = part1_train.loc[known, ["Efficacy", "Recruitment"]].to_numpy(dtype=int)
knn_er_neighbors = NearestNeighbors(n_neighbors=3).fit(X_train[known])
knn_er_neighbor_indices = knn_er_neighbors.kneighbors(X_train[er_unresolved], return_distance=False)
knn_er_vote_rates = original_er_labels[knn_er_neighbor_indices].mean(axis=1)
knn_er_recovered = (knn_er_vote_rates >= 0.30).astype(int)  # Independent multilabel threshold.
knn_er_labels = augmented[["Efficacy", "Recruitment"]].copy()
knn_er_labels.loc[er_unresolved, ["Efficacy", "Recruitment"]] = knn_er_recovered
assert knn_er_labels.isin([0, 1]).all().all()
assert np.array_equal(knn_er_labels.loc[~er_unresolved].to_numpy(), augmented.loc[~er_unresolved, ["Efficacy", "Recruitment"]].to_numpy())
assert np.array_equal(knn_er_labels.loc[known].to_numpy(), original_er_labels)

knn_er_estimators = list(safety_c01_reason_model.estimators_)
for j, reason in [(2, "Efficacy"), (3, "Recruitment")]:
    knn_er_estimators[j] = LogisticRegression(
        C=1.0, max_iter=500, solver="liblinear", random_state=RANDOM_STATE,
    ).fit(X_train, knn_er_labels[reason].to_numpy(dtype=int))
knn_er_reason_model = SelectiveReasonClassifier(knn_er_estimators)
knn_er_prediction_model = PreserveSFOFallbackClassifier(
    knn_er_reason_model, safety_c01_prediction_model
)
knn_er_validation_metrics = evaluate_reason_model(knn_er_prediction_model, X_val, part1_val)
reference_val_raw = safety_c01_reason_model.predict(X_val)
variant_val_raw = knn_er_reason_model.predict(X_val)
reference_val_final = predict_reason_labels(safety_c01_prediction_model, X_val)
variant_val_final = predict_reason_labels(knn_er_prediction_model, X_val)
reference_test_final = predict_reason_labels(safety_c01_prediction_model, X_test)
variant_test_final = predict_reason_labels(knn_er_prediction_model, X_test)
fixed_indices = [0, 1, 4]
knn_er_audit = pd.DataFrame({
    "Reason": [reasons[j] for j in fixed_indices],
    "Estimator reused": [knn_er_reason_model.estimators_[j] is safety_c01_reason_model.estimators_[j] for j in fixed_indices],
    "Raw validation changes": [int((reference_val_raw[:, j] != variant_val_raw[:, j]).sum()) for j in fixed_indices],
    "Final validation changes": [int((reference_val_final[:, j] != variant_val_final[:, j]).sum()) for j in fixed_indices],
    "Final test changes": [int((reference_test_final[:, j] != variant_test_final[:, j]).sum()) for j in fixed_indices],
})
assert knn_er_audit["Estimator reused"].all()
assert knn_er_audit[["Raw validation changes", "Final validation changes", "Final test changes"]].eq(0).all().all()
assert variant_val_final.any(axis=1).all() and variant_test_final.any(axis=1).all()
knn_er_coverage = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "Training rows": [1171, 1171],
    "Observed rows": [int(known.sum())] * 2,
    "Earlier cluster-recovered rows": [int((masked & ~er_unresolved).sum())] * 2,
    "KNN fallback rows": [int(er_unresolved.sum())] * 2,
    "KNN fallback positives": knn_er_recovered.sum(axis=0).tolist(),
    "KNN fallback negatives": (1 - knn_er_recovered).sum(axis=0).tolist(),
    "Final positive labels": [int(knn_er_labels[r].sum()) for r in ["Efficacy", "Recruitment"]],
    "Final negative labels": [int((knn_er_labels[r] == 0).sum()) for r in ["Efficacy", "Recruitment"]],
}).set_index("Reason")
knn_er_confusion = pd.DataFrame({
    "Reason": ["Efficacy", "Recruitment"],
    "TP": [int(((part1_val[r].to_numpy() == 1) & (variant_val_final[:, j] == 1)).sum()) for j, r in [(2, "Efficacy"), (3, "Recruitment")]],
    "FP": [int(((part1_val[r].to_numpy() == 0) & (variant_val_final[:, j] == 1)).sum()) for j, r in [(2, "Efficacy"), (3, "Recruitment")]],
    "FN": [int(((part1_val[r].to_numpy() == 1) & (variant_val_final[:, j] == 0)).sum()) for j, r in [(2, "Efficacy"), (3, "Recruitment")]],
}).set_index("Reason")
print("Safety C=.1 reference vs E/R KNN fallback validation:")
print(pd.DataFrame({"Safety C=.1 reference": safety_c01_validation_metrics, "E/R KNN fallback": knn_er_validation_metrics}).to_string())
print("E/R training coverage:")
print(knn_er_coverage.to_string())
print("Fixed S/F/O audit:")
print(knn_er_audit.to_string(index=False))
print("E/R validation confusion:")
print(knn_er_confusion.to_string())
print("E/R-only nonempty fallback trials on validation/test:", knn_er_prediction_model.er_fallback_count(X_val), knn_er_prediction_model.er_fallback_count(X_test))


In [ ]:
# Explicit export: K14 S/F/O blend with Safety C=.1 only; E/R remain K14 cluster-only.
# This selects the Safety-only experiment, not either E/R fallback experiment below it.
if "safety_c01_prediction_model" not in globals() or "safety_c01_validation_metrics" not in globals():
    raise ValueError("Run the K14 blend and Safety-only C=.1 cells first.")
if not np.isclose(safety_c01_validation_metrics["Sample-wise F1"], 0.9230769230769231):
    raise ValueError("Safety-only validation score differs from the selected model.")
safety_c01_submission_path = PROJECT_DIR / "prith_attempts" / "part_1" / "part1_kmeans_blend_sfo_k14_safety_c01_submission.csv"
safety_c01_submission = export_part1_predictions(
    safety_c01_prediction_model, X_test, part1_test, safety_c01_submission_path
)
assert len(safety_c01_submission) == len(part1_test) == 312
assert safety_c01_submission["ID"].equals(part1_test["ID"])
assert safety_c01_submission["Predicted"].notna().all()
assert safety_c01_submission["Predicted"].str.strip().ne("").all()
print("Saved:", safety_c01_submission_path)
print("Validation sample-wise F1:", safety_c01_validation_metrics["Sample-wise F1"])
print("Validation macro F1:", safety_c01_validation_metrics["Macro F1"])
print("Test trials:", len(safety_c01_submission))
print("Safety-only fallback trials:", safety_c01_prediction_model.safety_fallback_count(X_test))
safety_c01_submission.head()


In [ ]:
# Controlled K21 test: C=0.1 for all five independent reason classifiers.
# Compare with C=1 on identical labels, weights, and the same global nonempty fallback.
if "threshold30_reason_model" not in globals() or "threshold30_lineage" not in globals():
    raise ValueError("Run the K21 threshold30 quarter-weight blend cell first.")
all_c01_reasons = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
all_c01_selected = {"Safety", "Funding", "Other"}
all_c01_known = part1_train[all_c01_reasons].isin([0, 1]).all(axis=1).to_numpy()
assert all_c01_known.sum() == 157
all_c01_reference_augmented = threshold30_reference["part1_train_augmented"]
assert np.array_equal(
    all_c01_reference_augmented.loc[all_c01_known, all_c01_reasons].to_numpy(),
    part1_train.loc[all_c01_known, all_c01_reasons].to_numpy(),
)
all_c01_estimators = []
all_c01_coverage_rows = []
for reason in all_c01_reasons:
    if reason in all_c01_selected:
        labels = threshold30_lineage[reason].to_numpy(dtype=int)
        available = np.ones(len(part1_train), dtype=bool)
        weights = np.where(all_c01_known, 1.0, 0.25)
        assert np.array_equal(labels[all_c01_known], part1_train.loc[all_c01_known, reason].to_numpy(dtype=int))
    else:
        labels = all_c01_reference_augmented[reason].to_numpy(dtype=int)
        available = np.isin(labels, [0, 1])
        weights = np.ones(int(available.sum()))
        assert np.array_equal(labels[all_c01_known], part1_train.loc[all_c01_known, reason].to_numpy(dtype=int))
    estimator = LogisticRegression(
        C=0.1, max_iter=500, solver="liblinear", random_state=RANDOM_STATE,
    ).fit(X_train[available], labels[available], sample_weight=weights)
    all_c01_estimators.append(estimator)
    all_c01_coverage_rows.append({
        "Reason": reason,
        "Training rows": int(available.sum()),
        "Observed rows": int(all_c01_known.sum()),
        "Recovered rows": int((available & ~all_c01_known).sum()),
        "Unresolved rows": int((~available).sum()),
        "Recovered weight": 0.25 if reason in all_c01_selected else 1.0,
    })
all_c01_model = SelectiveReasonClassifier(all_c01_estimators)
all_c01_coverage = pd.DataFrame(all_c01_coverage_rows).set_index("Reason")
# Use the same five-reason argmax fallback for both models, isolating the C change.
all_c1_common_fallback_metrics = evaluate_reason_model(threshold30_reason_model, X_val, part1_val)
all_c01_common_fallback_metrics = evaluate_reason_model(all_c01_model, X_val, part1_val)
all_c1_fallback_val = int((~threshold30_reason_model.predict(X_val).any(axis=1)).sum())
all_c01_fallback_val = int((~all_c01_model.predict(X_val).any(axis=1)).sum())
all_c1_fallback_test = int((~threshold30_reason_model.predict(X_test).any(axis=1)).sum())
all_c01_fallback_test = int((~all_c01_model.predict(X_test).any(axis=1)).sum())
print("K21 common-fallback C comparison:")
print(pd.DataFrame({
    "Original C1 protected fallback": threshold30_validation_metrics,
    "C1 common fallback": all_c1_common_fallback_metrics,
    "C0.1 common fallback": all_c01_common_fallback_metrics,
}).to_string())
print("Per-reason training coverage and weights:")
print(all_c01_coverage.to_string())
print("Raw empty rows before common fallback, validation/test:",
      (all_c1_fallback_val, all_c1_fallback_test),
      (all_c01_fallback_val, all_c01_fallback_test))


In [ ]:
# Explicit export of all-five C=.1 K21 model using common five-reason fallback.
if "all_c01_model" not in globals() or "all_c01_common_fallback_metrics" not in globals():
    raise ValueError("Run the K21 all-five C=.1 regularization cell first.")
all_c01_submission_path = PROJECT_DIR / "prith_attempts" / "part_1" / "part1_kmeans_blend_sfo_k21_all_five_c01_submission.csv"
all_c01_submission = export_part1_predictions(
    all_c01_model, X_test, part1_test, all_c01_submission_path
)
assert len(all_c01_submission) == len(part1_test) == 312
assert all_c01_submission["ID"].equals(part1_test["ID"])
assert all_c01_submission["Predicted"].notna().all()
assert all_c01_submission["Predicted"].str.strip().ne("").all()
print("Saved:", all_c01_submission_path)
print("Validation sample-wise F1:", all_c01_common_fallback_metrics["Sample-wise F1"])
print("Validation macro F1:", all_c01_common_fallback_metrics["Macro F1"])
print("Test trials:", len(all_c01_submission))
print("Common fallback trials:", all_c01_fallback_test)
all_c01_submission.head()
